# 01 — Data Exploration

Load sample trials from the Willett handwriting dataset, inspect signal shapes, 
sampling rates, and label content, then visualize raw neural signals.

In [ ]:
import sys, os
sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline

from src.config import get_default_config
from src.data.loader import load_willett_dataset, build_trial_index_from_dir
from src.visualization.signal_plots import (
    plot_multichannel_timeseries,
    plot_channel_heatmap,
    plot_trial_overview,
)

## 1. Load configuration and trial index

In [ ]:
cfg = get_default_config()
print(f"Dataset : {cfg.dataset}")
print(f"Data path: {cfg.data_path}")
print(f"Subjects : {cfg.subjects}")

In [ ]:
# This will download + parse on first run, then use the cached index.
trial_index = load_willett_dataset(cfg)
print(f"Total trials: {len(trial_index)}")
trial_index.head(10)

## 2. Inspect trial shapes and labels

In [ ]:
print("Columns:", list(trial_index.columns))
print()
print("Timesteps stats:")
print(trial_index['n_timesteps'].describe())
print()
print("Channels stats:")
print(trial_index['n_channels'].describe())

In [ ]:
# Show first 5 labels
for i in range(min(5, len(trial_index))):
    row = trial_index.iloc[i]
    label_path = row['label_path']
    if label_path and os.path.exists(label_path):
        label = open(label_path).read().strip()
    else:
        label = '<no label>'
    print(f"Trial {row['trial_id']:4d}  |  T={row['n_timesteps']:5d}  C={row['n_channels']:3d}  |  label='{label}'")

## 3. Visualize sample trials

In [ ]:
N_SAMPLES = min(5, len(trial_index))

for i in range(N_SAMPLES):
    row = trial_index.iloc[i]
    sig = np.load(row['signal_path'])
    label = open(row['label_path']).read().strip() if row['label_path'] and os.path.exists(row['label_path']) else ''
    
    print(f"\n--- Trial {row['trial_id']}  shape={sig.shape}  label='{label}' ---")
    fig = plot_trial_overview(sig, label=label, fs=cfg.target_fs)
    plt.show()
    plt.close(fig)

## 4. Summary statistics

Verify that signal shapes and sampling rates match expectations from the Willett paper:
- 192 electrodes (Utah array)
- Spike-sorted data binned to ~200-250 Hz effective rate
- Character-level labels (a-z + space)

In [ ]:
# Channel distribution
print("Unique channel counts:", sorted(trial_index['n_channels'].unique()))
print("Unique subjects:", sorted(trial_index['subject'].unique()))
print("Unique sessions:", sorted(trial_index['session'].unique()))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
trial_index['n_timesteps'].hist(bins=30, ax=axes[0])
axes[0].set_title('Distribution of trial lengths')
axes[0].set_xlabel('Timesteps')

trial_index['n_channels'].hist(bins=20, ax=axes[1])
axes[1].set_title('Distribution of channel counts')
axes[1].set_xlabel('Channels')
plt.tight_layout()
plt.show()